In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta


# ============================================================
# 1. RETURN CALCULATIONS
# ============================================================

def one_period_returns(P: np.ndarray) -> np.ndarray:
    """
    Calculate one-period returns using the reversed-price convention.

    P[0] = most recent price
    P[1] = previous price
    etc.

    Therefore:

        R(t) = P[t] / P[t+1] - 1
    """
    return P[:-1] / P[1:] - 1


def momentum_stats(P: np.ndarray, S: int, T: int) -> dict:
    """
    Calculate momentum statistics for one stock.

    S = number of most recent periods to skip.
    T = number of periods used for the momentum calculation.

    Example:

        S = 1
        T = 6

    means:

        skip the most recent month
        use the previous 6 months
    """

    if len(P) < S + T + 1:
        return {
            "R_cum": np.nan,
            "R_mean": np.nan,
            "sigma": np.nan,
            "R_riskadj": np.nan
        }

    r = one_period_returns(P)

    # Select R(S) ... R(S+T-1)
    window = r[S:S + T]

    # Cumulative return across the T-period window
    R_cum = P[S] / P[S + T] - 1

    # Arithmetic mean of individual period returns
    R_mean = window.mean()

    # Sample variance
    sigma2 = ((window - R_mean) ** 2).sum() / (T - 1)

    sigma = np.sqrt(sigma2)

    # Risk-adjusted momentum
    R_riskadj = R_mean / sigma if sigma > 0 else np.nan

    return {
        "R_cum": R_cum,
        "R_mean": R_mean,
        "sigma": sigma,
        "R_riskadj": R_riskadj
    }


# ============================================================
# 2. DOWNLOAD EXACT 3-YEAR SAMPLE
# ============================================================

def download_prices(
    tickers: list[str],
    end_date: str,
    years: int = 3
) -> pd.DataFrame:
    """
    Download adjusted daily closing prices for exactly `years`
    before the specified end date.

    Example:

        end_date = "2026-07-31"
        years = 3

    gives approximately:

        2023-07-31 -> 2026-07-31

    `end_date` is the only date you need to change.
    """

    end = pd.Timestamp(end_date)

    # Calculate exact start date
    start = end - pd.DateOffset(years=years)

    print("=" * 70)
    print("DOWNLOAD SETTINGS")
    print("=" * 70)
    print(f"Requested end date   : {end.date()}")
    print(f"Requested start date : {start.date()}")
    print(f"Sample length        : {years} years")
    print("=" * 70)

    data = yf.download(
        tickers,
        start=start.strftime("%Y-%m-%d"),
        end=(end + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True
    )

    if data.empty:
        raise ValueError(
            "Yahoo Finance returned no data. "
            "Check your tickers and date range."
        )

    # --------------------------------------------------------
    # Handle yfinance's different return structures
    # --------------------------------------------------------

    if isinstance(data.columns, pd.MultiIndex):

        if "Close" in data.columns.get_level_values(0):
            data = data["Close"]

        elif "Close" in data.columns.get_level_values(1):
            data = data.xs("Close", axis=1, level=1)

        else:
            raise ValueError(
                "Could not find Close prices in yfinance response."
            )

    else:
        # Single ticker
        if "Close" in data.columns:
            data = data[["Close"]]
            data.columns = [tickers[0]]

    # Make sure dates are sorted
    data = data.sort_index()

    # Remove rows where EVERY ticker is missing
    data = data.dropna(how="all")

    # --------------------------------------------------------
    # Report actual downloaded range
    # --------------------------------------------------------

    if not data.empty:
        print(f"Actual first date   : {data.index[0].date()}")
        print(f"Actual last date    : {data.index[-1].date()}")
        print(f"Trading days        : {len(data)}")
        print(f"Tickers requested   : {len(tickers)}")
        print(f"Columns downloaded  : {len(data.columns)}")

    return data


# ============================================================
# 3. MONTHLY RESAMPLING
# ============================================================

def resample_monthly(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Convert daily prices into month-end prices.

    Each row represents the last available trading day
    of that calendar month.
    """

    monthly = prices.resample("ME").last()

    return monthly.dropna(how="all")

def rank_universe(
    prices: pd.DataFrame,
    S: int = 1,
    T: int = 6,
    min_history: int | None = None,) -> pd.DataFrame:
 
    table = []
    for ticker in prices.columns:
        series = prices[ticker].dropna()
        if len(series) < min_history:
            continue
 
        P = series.to_numpy()[::-1]
        stats = momentum_stats(P, S, T)
        stats["ticker"] = ticker
        table.append(stats)
 
    result = pd.DataFrame(table).set_index("ticker")
    result = result.sort_values("R_riskadj", ascending=False)
    result["rank"] = range(1, len(result) + 1)
 
    return result[["rank", "R_cum", "R_mean", "sigma", "R_riskadj"]]

